# Parallel phylogenetic tree inference: CUDA validation and benchmark

Start a fresh Kaggle notebook with an NVIDIA GPU and Internet access. Until the repositories are public, run `scripts/package_notebook_sources.sh`, upload the resulting ZIP as a Kaggle dataset, and attach it to this notebook. The standard run validates likelihood, marginal, MAP, and posterior-sampling inference in matched FP64 and FP32 host/CUDA builds, CPU CUDA-kernel emulation, native CUDA kernels, and (when available) Compute Sanitizer. It then times synthetic scaling cases, all 325 PANDIT 17.0 families with at least 100 tips, and the public 11,638-taxon Fish Tree of Life alignment against the matched-precision conventional CPU implementation and the pinned BEAGLE 4.0.1 CPU and CUDA implementations. Synthetic rows use 15 repetitions and empirical rows use three. JIT compilation is not involved. Fish Tree site batches increase geometrically up to the complete alignment or the capacity of each method; CUDA allocation failures and a cgroup-derived CPU memory guard stop only the affected sweep. The code-cell command may override `TREE_HMM_BENCHMARK_SECTIONS` with any subset of `validation synthetic fish pandit`, as well as the documented precision, repetition, sanitizer, and legacy `TREE_HMM_SKIP_*` controls. An earlier report may be attached separately or embedded as the second argument to `scripts/package_notebook_sources.sh`; all completed validation phases and benchmark rows are then reused automatically.

In [ ]:
from pathlib import Path
import subprocess
import zipfile

inputs = Path("/kaggle/input")
bundle = next(inputs.rglob("parallel_tree_inference_sources.zip"), None)
launcher = next(
    inputs.rglob("parallel_phylogenetic_inference/scripts/kaggle_cuda_notebook.sh"),
    None,
)

if bundle is not None:
    with zipfile.ZipFile(bundle) as archive:
        script = archive.read(
            "parallel_phylogenetic_inference/scripts/kaggle_cuda_notebook.sh"
        )
    subprocess.run(
        ["bash", "-s", "--", str(bundle)], input=script, check=True
    )
elif launcher is not None:
    source_root = launcher.parents[2]
    subprocess.run(["bash", str(launcher), str(source_root)], check=True)
else:
    raise FileNotFoundError(
        "Attach the parallel_tree_inference_sources input before running"
    )


The complete machine description, validation log, capacity boundaries, and benchmark CSV rows are checkpointed in `/kaggle/working/parallel_phylogenetics_cuda_report.txt`. Download that file even if a runtime interruption occurs; a later source bundle can embed it and resume only the missing work.